<h2> Pydantic - The game of structured outputs</h2>

making structured outputs for data validation following a given schema

the whole idea is to restrict the LLM response to a certain predefined schema with field validation.
 

In [18]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model(model="llama-3.3-70b-versatile",model_provider="groq",temperature="0.7")
                        
model


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x10e36fa70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10e37f860>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [29]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie"),
    director: str = Field(..., description="The director of the movie"),
    release_year: int = Field(..., description="The year the movie was released")
    
    # this will restrict the model to only return these three fields, and no additional text or explanation
    
    
    

In [23]:
smodel = model.with_structured_output(Movie)

In [21]:
smodel

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x10e36fa70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10e37f860>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'response_format': {'type': 'json_object', 'json_schema': {'properties': {'title': {'description': 'The title of the movie', 'title': 'Title', 'type': 'string'}, 'director': {'description': 'The director of the movie', 'title': 'Director', 'type': 'string'}, 'release_year': {'description': 'The year the movie was released', 'title': 'Release Year', 'type': 'integer'}}, 'required': ['title', 'director', 'release_year'], 'title': 'Movie', 'type':

In [24]:
smodel.invoke("What is the name of the movie directed by Christopher Nolan that was released in 2010?")

Movie(title='Inception', director='Christopher Nolan', release_year=2010)

In [26]:
model.invoke("What is the name of the movie directed by Christopher Nolan that was released in 2010? be descriptive")

AIMessage(content='The movie you\'re referring to is "Inception," a mind-bending, visually stunning science fiction action film written, co-produced, and directed by the acclaimed filmmaker Christopher Nolan. Released in 2010, "Inception" delves into the concept of shared dreaming, where a team of skilled thieves, led by Cobb (played by Leonardo DiCaprio), navigate the complexities of the human mind to plant an idea instead of stealing one.\n\nThe film boasts a talented ensemble cast, including Joseph Gordon-Levitt, Ellen Page, Tom Hardy, Ken Watanabe, Dileep Rao, Cillian Murphy, Tom Berenger, and Marion Cotillard. The movie\'s intricate plot, coupled with its innovative action sequences and clever use of non-linear storytelling, has made "Inception" a modern classic and a staple of contemporary cinema.\n\nWith its thought-provoking themes, impressive visual effects, and haunting score by Hans Zimmer, "Inception" has garnered widespread critical acclaim and has become one of Nolan\'s m

field validation is always applied means you have to declare the data type of the field you are mentioning 

In [27]:
zmodel= model.with_structured_output(Movie, include_raw=True)

zmodel.invoke("What is the name of the movie directed by Christopher Nolan that was released in 2010?")

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5k0t9k7xg', 'function': {'arguments': '{"director":"Christopher Nolan","release_year":2010,"title":"Inception"}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 275, 'total_tokens': 300, 'completion_time': 0.042710805, 'completion_tokens_details': None, 'prompt_time': 0.026357577, 'prompt_tokens_details': None, 'queue_time': 0.160839841, 'total_time': 0.069068382}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d1237-e968-7052-8d76-468fc2528518-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'release_year': 2010, 'title': 'Inception'}, 'id': '5k0t9k7xg', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 275, 'output_tokens': 25, 'total_toke

In [33]:
# nested structure 

from pydantic import BaseModel, Field

class Actor(BaseModel):
    
    name:str
    role: str


class MovieDetails (BaseModel):
    title: str = Field(..., description="The title of the movie")
    cast: list[Actor]
    director: str = Field(..., description="The director of the movie")
    genre: list[str]
    budget: int = Field(..., description="The budget of the movie in USD")
    
    release_year: int = Field(..., description="The year the movie was released")

In [34]:
zmodel=model.with_structured_output(MovieDetails)

zmodel.invoke("What is the name of the movie directed by Christopher Nolan that was released in 2010?"  )

MovieDetails(title='Inception', cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur')], director='Christopher Nolan', genre=['Action', 'Sci-Fi'], budget=160000000, release_year=2010)

<h2>TypedDict</h2>

In [42]:
from typing_extensions import Annotated,TypedDict

In [ ]:
class MovieDetails(TypedDict):
   title: Annotated[str,...,"the name of the movie"]
   cast: Annotated[list[dict],"the cast of actors with name and character"]
   director: Annotated[str,..., "the director"]             
   
   
   
tmodel=model.with_structured_output(MovieDetails)
tmodel.invoke("tell me about avengers endgame")

RuntimeError: no validator found for <class '__main__.Actor'>, see `arbitrary_types_allowed` in Config